In [ ]:
# 🧩 Scenario
# A smart classroom wants an AI assistant that can:
# Listen to student questions (speech)
# Analyze images (like diagrams/notes)
# Respond intelligently
# Each interaction must be tracked using Student ID.
# 🎯 Your Task
# Build a multimodal AI system integrating speech, vision, and language.

# ⚙️ Requirements
# You MUST use:
# Azure Speech Service
# Azure Computer Vision
# Azure OpenAI Service

# 🔧 Functional Flow
# Convert speech → text
# Analyze image → description
# Combine both → send to OpenAI
# Generate response with Student ID

In [ ]:
!pip install azure-cognitiveservices-speech
!pip install azure-ai-vision-imageanalysis
!pip install azure-core
!pip install openai

In [ ]:


import azure.cognitiveservices.speech as speechsdk
from azure.core.credentials import AzureKeyCredential
from azure.ai.vision.imageanalysis import ImageAnalysisClient
from azure.ai.vision.imageanalysis.models import VisualFeatures
from openai import OpenAI
from google.colab import userdata

# 🔐 CONFIGURATION (Replace with your values)

# 🔊 Azure Speech
SPEECH_KEY = userdata.get("SPEECH_API")
SPEECH_REGION = "koreacentral"

# 🖼️ Azure Vision
VISION_ENDPOINT = "https://mrinal-vision.cognitiveservices.azure.com/"
VISION_KEY=userdata.get("VISION_ENDPOINT")

# 🤖 Azure OpenAI
OPENAI_ENDPOINT="https://mrinal06022005.openai.azure.com/"
OPENAI_API_KEY = userdata.get("AZURE_OPENAPI_KEY")
DEPLOYMENT_NAME = "gpt-4.1"

# 🚀 Initialize Clients

# Vision Client
vision_client = ImageAnalysisClient(
    endpoint=VISION_ENDPOINT,
    credential=AzureKeyCredential(VISION_KEY)
)

# OpenAI Client
openai_client = OpenAI(
    api_key=OPENAI_API_KEY,
    base_url=f"{OPENAI_ENDPOINT}/openai/deployments/{DEPLOYMENT_NAME}/",
    default_query={"api-version": "2024-02-01"}
)


#  Speech → Text
def speech_to_text(audio_file):

    speech_config = speechsdk.SpeechConfig(
        subscription=SPEECH_KEY,
        region=SPEECH_REGION
    )

    speech_config.speech_recognition_language = "en-US"

    audio_config = speechsdk.audio.AudioConfig(filename=audio_file)

    recognizer = speechsdk.SpeechRecognizer(
        speech_config=speech_config,
        audio_config=audio_config
    )

    result = recognizer.recognize_once()

    if result.reason == speechsdk.ResultReason.RecognizedSpeech:
        return result.text

    elif result.reason == speechsdk.ResultReason.NoMatch:
        return "No speech recognized."

    elif result.reason == speechsdk.ResultReason.Canceled:
        cancellation = result.cancellation_details
        return f"Speech canceled: {cancellation.reason}"

    return "Error in speech recognition"


# 🖼️ Image → Description + OCR

def analyze_image(image_path):

    with open(image_path, "rb") as f:
        image_data = f.read()

    result = vision_client.analyze(
        image_data=image_data,
        visual_features=[
            VisualFeatures.CAPTION,
            VisualFeatures.READ
        ]
    )

    caption = ""
    text_data = []

    # Caption
    if result.caption:
        caption = result.caption.text

    # OCR text
    if result.read:
        for block in result.read.blocks:
            for line in block.lines:
                text_data.append(line.text)

    return f"Caption: {caption}\nDetected Text: {' | '.join(text_data)}"

# 🤖 Final AI Assistant
def smart_assistant(student_id, audio_file, image_file):

    print("\n🔊 Converting speech to text...")
    speech_text = speech_to_text(audio_file)

    print("🖼️ Analyzing image...")
    image_info = analyze_image(image_file)

    print("🤖 Generating AI response...")

    prompt = f"""
    You are a smart classroom assistant.

    Student ID: {student_id}

    Student Question:
    {speech_text}

    Image Analysis:
    {image_info}

    Instructions:
    - Understand both speech and image
    - Give a clear explanation
    - Start answer with: [Student ID: {student_id}]
    """

    response = openai_client.chat.completions.create(
        model=DEPLOYMENT_NAME,
        messages=[
            {"role": "system", "content": "You are a helpful teaching assistant."},
            {"role": "user", "content": prompt}
        ],
        temperature=0.5
    )

    return response.choices[0].message.content


# ▶️ RUN PROGRAM
if __name__ == "__main__":

    # 📥 Input
    student_id = "BTECH2026_333"
    audio_file = "/content/speech.wav"
    image_file = "/content/imagee.png"

    # 🚀 Run assistant
    output = smart_assistant(student_id, audio_file, image_file)

    # 📤 Output
    print("\n FINAL RESPONSE:\n")
    print(output)


🔊 Converting speech to text...
🖼️ Analyzing image...
🤖 Generating AI response...

 FINAL RESPONSE:

[Student ID: BTECH2026_333]

The diagram shows the structure of a neuron in a deep learning model, which is inspired by the neurons in the human brain. Here’s what each part means:

- Input (features): These are the initial data or information that the model receives. For example, in image recognition, the input could be pixel values.
- Hidden Layers: These are layers between the input and output. Each layer contains many neurons, and the diagram notes "lots of layers," which is why it's called "deep learning." These layers help the model learn complex patterns from the data.
- Output (prediction): This is the final result or prediction made by the model after processing the inputs through the hidden layers. For example, it could predict whether an image contains a cat or not.

In summary, the diagram illustrates how data flows through a deep learning model: starting from input features